# LSTM for Jena Climate Prediction

In this exercise we use an **LSTM** to predict temperature from the Jena Climate dataset, a multi-variate weather time series.

**Dataset:** `jena_climate.csv` -- Weather measurements recorded every 10 minutes (temperature, pressure, humidity, wind, etc.).

**Learning objectives:**
1. Explore a multi-variate time series dataset.
2. Split in time, fit the scaler on training rows only and build windows separately inside each split.
3. Score persistence and 24-hour seasonal-naive baselines on the same windows as the model.
4. Build an LSTM that uses multiple input features to predict temperature, with early stopping on validation loss.
5. Compare single-feature vs multi-feature prediction on validation, then score the test windows once.

**Evaluation design.** Training, validation and test are consecutive periods (70% / 20% / 10%). Every choice -- the stopping epoch, single- vs multi-feature inputs -- is made on validation. The test windows are scored once, in Section 11, after those choices are locked.

**Runtime.** About 4 minutes for the whole solution on an Apple M3 Max laptop CPU: 2.5 for the main model and 1.5 for the optional single-feature model in Section 10; allow up to twice as long on older laptops. To keep it there, each training epoch uses a random quarter of the training windows (Section 6) and training stops after at most 15 epochs.

---
## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)  # initial weights, dropout and the random training batches

print(f"PyTorch version: {torch.__version__}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

---
## 2. Load and Explore the Data

In [ ]:
df = pd.read_csv("jena_climate.csv")
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df.head()

In [ ]:
df.describe()

In [ ]:
print(f"Date range: {df.iloc[0, 0]} to {df.iloc[-1, 0]}")
print(f"Missing values:\n{df.isnull().sum()}")

---
## 3. Visualize Temperature Over Time

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Full temperature series
axes[0].plot(df["T (degC)"].values, linewidth=0.5)
axes[0].set_title("Temperature Over Time (Full Dataset)")
axes[0].set_ylabel("T (degC)")
axes[0].grid(True, alpha=0.3)

# Zoom in on first 2000 samples (~2 weeks)
axes[1].plot(df["T (degC)"].values[:2000], linewidth=0.8)
axes[1].set_title("Temperature (First ~2 Weeks)")
axes[1].set_xlabel("Sample Index (10-min intervals)")
axes[1].set_ylabel("T (degC)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 4. Subsample and Select Features

The original data is recorded every 10 minutes. We subsample to hourly intervals (every 6th row) to reduce the dataset size and speed up training. The file contains a few duplicated time stamps and gaps, so 7 of the 70,091 hourly steps are irregular; the largest are a 17-hour gap in September 2014 (validation period) and a 3-day gap in October 2016 (test period). Windows that span such a step join hours that are not neighbours in time; about 1% of the windows are affected, and we accept this simplification here.

`df.describe()` shows a minimum of -9999 in `wv (m/s)` and `max. wv (m/s)`. That is a logger error code, not a wind speed. We treat it as missing and carry the previous valid reading forward: a forward fill uses past values only, so it is available at every forecast origin and needs no fitting.

In [ ]:
# Subsample to hourly
df_hourly = df.iloc[::6].reset_index(drop=True)
print(f"Hourly shape: {df_hourly.shape}")

# Replace the -9999 error code in the wind columns by the previous valid reading
for col in ["wv (m/s)", "max. wv (m/s)"]:
    bad = df_hourly[col] < 0
    print(f"{col}: {bad.sum()} hourly rows with the error code")
    df_hourly[col] = df_hourly[col].mask(bad).ffill()

# Select features for prediction
feature_cols = ["p (mbar)", "T (degC)", "Tpot (K)", "Tdew (degC)",
                "rh (%)", "VPmax (mbar)", "VPact (mbar)", "VPdef (mbar)",
                "sh (g/kg)", "H2OC (mmol/mol)", "rho (g/m**3)",
                "wv (m/s)", "max. wv (m/s)", "wd (deg)"]

target_col = "T (degC)"

data = df_hourly[feature_cols].values.astype(np.float32)
times = pd.to_datetime(df_hourly["Date Time"], format="%d.%m.%Y %H:%M:%S")
print(f"Feature matrix shape: {data.shape}")
print(f"Target column: {target_col}")

---
## 5. Normalize Features

Split first -- 70% training, 20% validation, 10% test, in time order -- then fit the scaler on the training rows only and apply it unchanged to validation and test.

In [ ]:
# Split first, then normalize
n = len(data)
train_end = int(n * 0.7)
val_end = int(n * 0.9)

train_data = data[:train_end]
val_data = data[train_end:val_end]
test_data = data[val_end:]

print(f"Train: {train_data.shape}, Val: {val_data.shape}, Test: {test_data.shape}")
for name, start, stop in [("Train", 0, train_end), ("Val", train_end, val_end), ("Test", val_end, n)]:
    print(f"{name:>5}: {times.iloc[start]:%Y-%m-%d %H:%M} to {times.iloc[stop - 1]:%Y-%m-%d %H:%M}")

# Fit scaler on training data only
scaler = StandardScaler()
train_data = scaler.fit_transform(train_data)
val_data = scaler.transform(val_data)
test_data = scaler.transform(test_data)

# Find the index of the target column in our feature list
target_idx = feature_cols.index(target_col)
print(f"Target column index: {target_idx}")

---
## 6. Create Sequences

We use the past 72 hours (72 samples at hourly resolution) to predict the temperature 12 hours ahead. Windows are built separately inside each split, so no training window has an input or a target in the validation period; the cost is a few lost windows at each boundary.

In [ ]:
def create_sequences(data, input_len, forecast_horizon, target_idx):
    """Create multi-variate input sequences and single-step target values.
    
    Args:
        data: array of shape (time_steps, num_features)
        input_len: number of past hours to use
        forecast_horizon: how many steps ahead to predict
        target_idx: column index of the target variable
    """
    X, y = [], []
    for i in range(len(data) - input_len - forecast_horizon + 1):
        X.append(data[i : i + input_len])  # all features
        y.append(data[i + input_len + forecast_horizon - 1, target_idx])  # target only
    return np.array(X), np.array(y).reshape(-1, 1)


INPUT_LEN = 72  # past 72 hours
FORECAST_HORIZON = 12  # predict 12 hours ahead

X_train, y_train = create_sequences(train_data, INPUT_LEN, FORECAST_HORIZON, target_idx)
X_val, y_val = create_sequences(val_data, INPUT_LEN, FORECAST_HORIZON, target_idx)
X_test, y_test = create_sequences(test_data, INPUT_LEN, FORECAST_HORIZON, target_idx)

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape},   y_val:   {y_val.shape}")
print(f"X_test:  {X_test.shape},  y_test:  {y_test.shape}")

# Alignment check on the first test window: the target is 12 hours after the last input
print(f"First test window: last input {times.iloc[val_end + INPUT_LEN - 1]}, "
      f"target {times.iloc[val_end + INPUT_LEN + FORECAST_HORIZON - 1]}")

### Baselines on the same windows

A model is only useful if it beats simple forecasts scored on **exactly the same targets**. Both baselines are read from the input windows, so there is one forecast per window:
- **Persistence**: the temperature at the origin, `X[:, -1, target_idx]`.
- **24-hour seasonal naive**: the temperature at the same clock hour one day before the target. The target is 72 + 12 - 1 = 83 steps after the window start, so the value 24 hours earlier is inside the window at index 83 - 24 = 59 (index 47 would be 24 hours before the origin, not before the target).

Errors are computed in degC after inverting the scaling with the target column's mean and standard deviation. We score the validation windows now; the test windows are scored once, in Section 11.

In [ ]:
target_mean = scaler.mean_[target_idx]
target_std = scaler.scale_[target_idx]
SEASONAL_IDX = INPUT_LEN + FORECAST_HORIZON - 1 - 24  # = 59


def to_degc(z):
    """Invert the StandardScaler for the target column."""
    return np.asarray(z).reshape(-1) * target_std + target_mean


def score(y_true, y_pred):
    """MAE and RMSE in degC."""
    return {"MAE (degC)": mean_absolute_error(y_true, y_pred),
            "RMSE (degC)": np.sqrt(mean_squared_error(y_true, y_pred))}


def baseline_forecasts(X):
    """Persistence and 24-hour seasonal naive, one forecast per window, in degC."""
    return {"Persistence": to_degc(X[:, -1, target_idx]),
            "Seasonal naive (24 h)": to_degc(X[:, SEASONAL_IDX, target_idx])}


y_val_degc = to_degc(y_val)
val_scores = {name: score(y_val_degc, pred) for name, pred in baseline_forecasts(X_val).items()}
print(f"Validation windows: {len(y_val_degc)}")
pd.DataFrame(val_scores).T.round(2)

In [ ]:
# Convert to tensors and DataLoaders
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32)

BATCH_SIZE = 256
# Classroom budget: each epoch draws a fresh random sample of 12,288 of the 48,981
# training windows (about a quarter). Neighbouring windows share 71 of their 72 hours,
# so little information is lost. Set TRAIN_WINDOWS_PER_EPOCH = len(train_ds) for full epochs.
TRAIN_WINDOWS_PER_EPOCH = 12_288
train_ds = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          sampler=RandomSampler(train_ds, num_samples=TRAIN_WINDOWS_PER_EPOCH))
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t),
                        batch_size=1024, shuffle=False)  # no gradients: larger batches are fine

print(f"Train batches per epoch: {len(train_loader)}, Val batches: {len(val_loader)}")

---
## 7. Build the Multi-Variate LSTM Model

In [ ]:
class ClimateLSTM(nn.Module):
    """LSTM for multi-variate time series forecasting."""

    def __init__(self, input_size, hidden_size=128, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_hidden = lstm_out[:, -1, :]
        return self.fc(last_hidden)


num_features = X_train.shape[2]
model = ClimateLSTM(input_size=num_features).to(DEVICE)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

---
## 8. Train with Early Stopping

Early stopping watches the **validation** loss. When it has not improved for `PATIENCE` epochs, training stops, and the weights from the best epoch -- saved as a deep copy, so later updates cannot change them -- are restored before any evaluation. The test windows play no part in this choice. The loop is wrapped in a function so that the optional single-feature model in Section 10 is trained with exactly the same procedure.

In [ ]:
import copy


def train_with_early_stopping(model, train_loader, val_loader, num_epochs, patience, lr=1e-3):
    """Train with MSE loss; stop when the validation loss has not improved for
    `patience` epochs and restore the best weights (a deep copy)."""
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    train_losses, val_losses = [], []
    best_val_loss, best_epoch, best_state = float("inf"), 0, None
    patience_counter = 0

    for epoch in range(1, num_epochs + 1):
        # --- Train ---
        model.train()
        running_loss, n_seen = 0.0, 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * X_batch.size(0)
            n_seen += X_batch.size(0)
        train_losses.append(running_loss / n_seen)

        # --- Validate ---
        model.eval()
        val_running = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
                val_running += criterion(model(X_batch), y_batch).item() * X_batch.size(0)
        val_losses.append(val_running / len(val_loader.dataset))

        # Early stopping check (validation loss only)
        if val_losses[-1] < best_val_loss:
            best_val_loss, best_epoch = val_losses[-1], epoch
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{num_epochs}  "
                  f"Train: {train_losses[-1]:.6f}  Val: {val_losses[-1]:.6f}  "
                  f"Patience: {patience_counter}/{patience}")

        if patience_counter >= patience:
            print(f"\nEarly stopping at epoch {epoch}!")
            break

    # Restore the best model before any evaluation
    model.load_state_dict(best_state)
    print(f"Restored epoch {best_epoch} with val loss: {best_val_loss:.6f}")
    return train_losses, val_losses, best_epoch


NUM_EPOCHS = 15
PATIENCE = 5
train_losses, val_losses, best_epoch = train_with_early_stopping(
    model, train_loader, val_loader, NUM_EPOCHS, PATIENCE)

---
## 9. Plot Loss Curves

In [ ]:
epochs = range(1, len(train_losses) + 1)
plt.figure(figsize=(8, 4))
plt.plot(epochs, train_losses, label="Train Loss")
plt.plot(epochs, val_losses, label="Validation Loss")
plt.axvline(best_epoch, color="gray", linestyle="--", label=f"Chosen epoch ({best_epoch})")
plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE (scaled units)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 10. Compare: Single-Feature vs Multi-Feature (validation)

Comparisons that could change our choices use the **validation** windows only. First the 14-feature model next to the baselines; then, optionally, a second model that only uses past temperature values (single feature). If time is short, skip the two optional cells and go to Section 11.

In [ ]:
def predict_degc(model, X, batch_size=1024):
    """Forecasts in degC for an array of windows."""
    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            batch = torch.tensor(X[i : i + batch_size], dtype=torch.float32).to(DEVICE)
            preds.append(model(batch).cpu().numpy())
    return to_degc(np.concatenate(preds))


val_scores["LSTM, 14 features"] = score(y_val_degc, predict_degc(model, X_val))
pd.DataFrame(val_scores).T.round(2)

In [ ]:
# Optional: single-feature windows (temperature only). They are the same windows
# restricted to the temperature column, so the targets are identical.
X_train_sf = X_train[:, :, [target_idx]]
X_val_sf = X_val[:, :, [target_idx]]
print(f"Single-feature X_train: {X_train_sf.shape}")

# DataLoaders
sf_train_ds = TensorDataset(torch.tensor(X_train_sf, dtype=torch.float32), y_train_t)
sf_train_loader = DataLoader(sf_train_ds, batch_size=BATCH_SIZE,
                             sampler=RandomSampler(sf_train_ds, num_samples=TRAIN_WINDOWS_PER_EPOCH))
sf_val_loader = DataLoader(
    TensorDataset(torch.tensor(X_val_sf, dtype=torch.float32), y_val_t),
    batch_size=1024, shuffle=False)

In [ ]:
# Optional: train the single-feature model with the same early-stopping procedure
sf_model = ClimateLSTM(input_size=1, hidden_size=64).to(DEVICE)
sf_train_losses, sf_val_losses, sf_best_epoch = train_with_early_stopping(
    sf_model, sf_train_loader, sf_val_loader, NUM_EPOCHS, PATIENCE)

# Compare on validation
val_scores["LSTM, temperature only"] = score(y_val_degc, predict_degc(sf_model, X_val_sf))
mf_mae = val_scores["LSTM, 14 features"]["MAE (degC)"]
sf_mae = val_scores["LSTM, temperature only"]["MAE (degC)"]
print(f"\nOn validation, using multiple features {'improves' if mf_mae < sf_mae else 'does not improve'} "
      f"prediction (MAE {mf_mae:.2f} vs {sf_mae:.2f} degC).")
pd.DataFrame(val_scores).T.round(2)

---
## 11. Evaluate on Test Set

This is the final evaluation, run once after every choice above has been made on validation. All methods are scored on the same 6,927 test windows (identical target time stamps), baselines first. If the optional single-feature model was trained, it is reported too, but the choice between the two LSTMs was made on validation in Section 10; do not switch models because of this table.

In [ ]:
y_test_degc = to_degc(y_test)

test_forecasts = baseline_forecasts(X_test)
test_forecasts["LSTM, 14 features"] = predict_degc(model, X_test)
if "sf_model" in globals():  # only if the optional single-feature model was trained
    test_forecasts["LSTM, temperature only"] = predict_degc(sf_model, X_test[:, :, [target_idx]])

test_scores = {name: score(y_test_degc, pred) for name, pred in test_forecasts.items()}
print(f"Test windows: {len(y_test_degc)}; standard deviation of the target: {y_test_degc.std():.2f} degC")
pd.DataFrame(test_scores).T.round(2)

In [ ]:
# One test week (168 consecutive target hours): actual, LSTM and the seasonal naive
target_times = times.iloc[val_end + INPUT_LEN + FORECAST_HORIZON - 1 : n].reset_index(drop=True)
week = slice(0, 168)  # change the start to look at other weeks

plt.figure(figsize=(14, 5))
plt.plot(target_times.iloc[week], y_test_degc[week], label="Actual", linewidth=1.2)
plt.plot(target_times.iloc[week], test_forecasts["LSTM, 14 features"][week], label="LSTM, 14 features", linewidth=1.0)
plt.plot(target_times.iloc[week], test_forecasts["Seasonal naive (24 h)"][week], label="Seasonal naive (24 h)",
         linewidth=1.0, linestyle="--")
plt.title("Temperature 12 hours ahead: one test week")
plt.xlabel("Target time")
plt.ylabel("Temperature (degC)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Where does the LSTM fail? Days with the largest mean absolute error
daily_mae = pd.Series(np.abs(test_forecasts["LSTM, 14 features"] - y_test_degc),
                      index=target_times).resample("D").mean()
print("Test days with the largest LSTM MAE (degC):")
print(daily_mae.nlargest(3).round(2))

---
## 12. Summary

In this notebook we:

1. Loaded the Jena Climate dataset -- a rich multi-variate time series of weather measurements.
2. Subsampled to hourly resolution, replaced the -9999 wind error code and selected the features.
3. Split 70/20/10 in time and normalized features using `StandardScaler` (fit on train only).
4. Created sequences separately inside each split: 72 hours of input to predict temperature 12 hours ahead.
5. Scored persistence and the 24-hour seasonal naive on the same windows.
6. Built a multi-variate LSTM with a two-layer FC head and trained it with early stopping on validation loss.
7. Compared single-feature vs multi-feature models on validation.
8. Scored the test windows once: both baselines first, then the LSTM.

**Key takeaways:**
- At 12 hours ahead persistence is a weak baseline (the opposite phase of the daily cycle); the 24-hour seasonal naive is the fair reference.
- Early stopping on validation loss prevents overfitting and saves training time; the test set plays no part in it.
- Multi-variate inputs can provide additional context -- check on validation whether they actually help.
- Always split time series chronologically (not randomly), fit preprocessing on the training period and build windows per split to avoid data leakage.